# Notebook 2 — Data preparation

Goal: turn the messy strings from notebook 1 (`'CHF 2'500.–'`, `'3.5 Zimmer'`, `'85 m²'`) into typed pandas columns, drop missing/invalid rows, deduplicate, and persist the result.

**Rubric coverage:**
- ✅ #2 Data preparation incl. regular expressions
- ✅ #3 pandas + Python data structures (lists, dicts, sets, tuples)
- ✅ #4 Conditional statements + loops + `break`/`continue`
- ✅ #5 Procedural and OOP — we both call functions and instantiate the `Listing` / `RentalMarket` classes
- ✅ Bonus #3 SQLite database + SQL queries

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import logging
import pandas as pd
logging.basicConfig(level=logging.INFO, format='%(message)s')

In [ ]:
raw = pd.read_csv(ROOT / 'data' / 'listings_raw.csv')
canton_ref = pd.read_csv(ROOT / 'data' / 'canton_reference.csv')
print(f'Raw listings: {len(raw):,}')
raw.head(3)

## 1. Regex parsers — bonus #2 / minimum #2

We extract numbers from Swiss-formatted strings. Three patterns reused throughout the project:

In [ ]:
from app.data_cleaning import parse_price, parse_rooms, parse_area

# A short list (built-in data structure) of (text, parser) tuples
samples: list[tuple[str, callable]] = [
    ("CHF 2'500.–", parse_price),
    ("CHF 1'200.50", parse_price),
    ("3.5 Zimmer",    parse_rooms),
    ("85 m²",         parse_area),
    ("Auf Anfrage",   parse_price),  # parser must return None gracefully
]

for text, fn in samples:
    print(f"{fn.__name__:>12s}({text!r:>16}) -> {fn(text)!r}")

## 2. Full cleaning pipeline

The function `clean_listings` chains the regex parsers, drops invalid rows in a `for` loop with `break`, deduplicates by `listing_id` (a `set` of seen IDs internally), and adds derived columns (`price_per_m2`, `is_urban`).

In [ ]:
from app.data_cleaning import clean_listings, attach_canton_reference

clean = clean_listings(raw)
print(f'Clean listings: {len(clean):,}  ({len(clean) / len(raw):.0%} of raw)')
clean.head(3)

## 3. Joining the BFS reference

We merge each listing with the canton-level BFS averages, producing an `expected_rent_bfs` column (BFS mean × m²) and a `rent_gap_chf` column (asking minus expected).

In [ ]:
enriched = attach_canton_reference(clean, canton_ref)
enriched[['canton', 'rooms', 'living_space_m2', 'rent_chf',
          'expected_rent_bfs', 'rent_gap_chf']].head()

## 4. OOP — `Listing` and `RentalMarket`

Same data, different paradigm. The procedural pipeline above operated on a DataFrame; here we build a `RentalMarket` of `Listing` objects so we can use methods like `average_price_per_m2()` and `by_canton()`.

In [ ]:
from app.models import market_from_dataframe

market = market_from_dataframe(enriched)
print('Market summary:', market.summary())
print('First listing:', market.listings[0])

In [ ]:
# `set` of unique cantons present
unique_cantons = market.cantons()
print(f'Cantons in market: {sorted(unique_cantons)}')

# Per-canton aggregation via dict
by_canton = market.by_canton()
for code in sorted(by_canton)[:5]:
    sub = by_canton[code]
    print(f'  {code}: {len(sub):>3d} listings, '
          f'avg CHF {sub.average_rent():,.0f}/month'.replace(',', "'"))

## 5. Persist into SQLite (bonus #3)

We materialise the cleaned data plus canton reference in a SQLite database and run a sample SQL query to confirm everything wrote correctly.

In [ ]:
from app import database

database.init_schema()
database.write_canton_stats(canton_ref)
database.write_listings(enriched)

print(f'Database at: {database.DEFAULT_DB_PATH.relative_to(ROOT)}')

In [ ]:
# Sample SQL query: average rent by canton, joined with population
database.query_avg_rent_by_canton().head(10)

In [ ]:
# Another SQL query: distribution by room count
database.query_room_distribution()

In [ ]:
# Save the enriched frame too — handy for notebook 3 without re-running
enriched.to_csv(ROOT / 'data' / 'listings_clean.csv', index=False)
print('Saved listings_clean.csv')

Continue in **`03_analysis_visualization.ipynb`**.